# HealthAlliance DataSpace — Patient Readmission Risk: Exploratory Analysis

This notebook covers:
1. Loading and inspecting the patient dataset
2. Exploratory Data Analysis (EDA)
3. Feature engineering
4. Model training and evaluation
5. Feature importance visualisation

**Dataset**: Synthetic patient data from three partner institutions (DKFZ, UKHD, EMBL) — generated to preserve privacy while reflecting realistic clinical risk patterns.

**Target variable**: `readmitted` — 1 if the patient was re-hospitalised within 30 days, 0 otherwise.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # project root

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import RocCurveDisplay, ConfusionMatrixDisplay

from src.data import generate_training_data
from src.models import train_model, predict_risk

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110
print("Imports OK")

## 1 — Load Dataset

In [ ]:
df = generate_training_data(n_patients=1000, seed=42)
print(f"Shape: {df.shape}")
df.head(10)

In [ ]:
df.describe().T.round(2)

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nClass balance:\n{df['readmitted'].value_counts(normalize=True).round(3)}")

## 2 — Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
features = ["age", "num_conditions", "num_medications", "recent_encounters", "gender_encoded"]
colors = ["#2ecc71", "#e74c3c"]  # green = not readmitted, red = readmitted

for ax, feat in zip(axes.flat, features):
    for label, color in zip([0, 1], colors):
        subset = df[df["readmitted"] == label][feat]
        ax.hist(subset, bins=20, alpha=0.6, color=color,
                label="Not readmitted" if label == 0 else "Readmitted")
    ax.set_title(feat, fontsize=11)
    ax.set_xlabel(feat)
    ax.set_ylabel("Count")

handles = [mpatches.Patch(color=c, label=l)
           for c, l in zip(colors, ["Not readmitted", "Readmitted"])]
axes.flat[-1].legend(handles=handles)
axes.flat[-1].set_visible(True)  # keep the legend cell

fig.suptitle("Feature Distributions by Readmission Status", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(7, 5))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdYlGn",
            vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# Readmission rate by age group
df["age_group"] = pd.cut(df["age"], bins=[0, 40, 55, 65, 75, 100],
                          labels=["<40", "40-55", "55-65", "65-75", ">75"])
rate_by_age = df.groupby("age_group")["readmitted"].mean().reset_index()

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(rate_by_age["age_group"].astype(str), rate_by_age["readmitted"],
               color=sns.color_palette("flare", len(rate_by_age)))
ax.set_title("30-Day Readmission Rate by Age Group")
ax.set_xlabel("Age Group")
ax.set_ylabel("Readmission Rate")
ax.set_ylim(0, 1)
for bar, val in zip(bars, rate_by_age["readmitted"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"{val:.0%}", ha="center", fontsize=10)
plt.tight_layout()
plt.show()

## 3 — Model Training

In [ ]:
model, scaler, metrics = train_model(df.drop(columns=["age_group"]))

print(f"ROC-AUC: {metrics['roc_auc']:.4f}")
print()
print(metrics["classification_report"])

## 4 — Feature Importance

In [ ]:
importance = pd.Series(metrics["feature_importance"]).sort_values()

fig, ax = plt.subplots(figsize=(7, 4))
importance.plot.barh(ax=ax, color=sns.color_palette("Blues_d", len(importance)))
ax.set_title("RandomForest Feature Importance")
ax.set_xlabel("Importance (Gini)")
for patch in ax.patches:
    ax.text(patch.get_width() + 0.002, patch.get_y() + patch.get_height() / 2,
            f"{patch.get_width():.3f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

## 5 — ROC Curve & Confusion Matrix

In [ ]:
from sklearn.model_selection import train_test_split

feature_cols = ["age", "num_conditions", "num_medications", "recent_encounters", "gender_encoded"]
X = df[feature_cols].fillna(0)
y = df["readmitted"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                     random_state=42, stratify=y)
X_test_scaled = scaler.transform(X_test)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

RocCurveDisplay.from_estimator(model, X_test_scaled, y_test, ax=ax1)
ax1.plot([0, 1], [0, 1], "k--", lw=1)
ax1.set_title("ROC Curve — Readmission Risk Model")

ConfusionMatrixDisplay.from_estimator(model, X_test_scaled, y_test, ax=ax2,
                                      display_labels=["Not Readmitted", "Readmitted"],
                                      cmap="Blues")
ax2.set_title("Confusion Matrix")

plt.tight_layout()
plt.show()

## 6 — Sample Predictions

In [ ]:
sample_patients = [
    {"label": "High-risk elderly",   "age": 80, "num_conditions": 4, "num_medications": 7, "recent_encounters": 6, "gender_encoded": 1},
    {"label": "Moderate-risk adult", "age": 60, "num_conditions": 2, "num_medications": 3, "recent_encounters": 2, "gender_encoded": 0},
    {"label": "Low-risk young",      "age": 28, "num_conditions": 0, "num_medications": 0, "recent_encounters": 0, "gender_encoded": 1},
    {"label": "DKFZ-001 profile",    "age": 74, "num_conditions": 3, "num_medications": 6, "recent_encounters": 6, "gender_encoded": 1},
    {"label": "EMBL-004 profile",    "age": 88, "num_conditions": 4, "num_medications": 7, "recent_encounters": 9, "gender_encoded": 0},
]

rows = []
for p in sample_patients:
    features = {k: v for k, v in p.items() if k != "label"}
    risk = predict_risk(model, scaler, features)
    level = "HIGH" if risk >= 0.6 else "MEDIUM" if risk >= 0.3 else "LOW"
    rows.append({"Patient Profile": p["label"], "Risk Score": round(risk, 3), "Risk Level": level})

pd.DataFrame(rows).set_index("Patient Profile")